# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


In [1]:
class matrix:
    def __init__(self, *args):
        if len(args) == 2 and all(isinstance(x, int) for x in args):
            n, m = args
            if n <= 0 or m <= 0:
                raise ValueError("Matrix dimensions must be positive integers.")
            self.n = n
            self.m = m
            self.data = [[0 for _ in range(m)] for _ in range(n)]

        elif len(args) == 1 and isinstance(args[0], list):
            data = args[0]

            if not data or not all(isinstance(row, list) for row in data):
                raise ValueError("Matrix must be initialized with a list of lists.")

            row_length = len(data[0])
            for row in data:
                if len(row) != row_length:
                    raise ValueError("All rows must have the same number of columns.")

            self.n = len(data)
            self.m = row_length
            self.data = [row[:] for row in data]  # Deep copy

        else:
            raise ValueError("Invalid initialization.")

    def __getitem__(self, key):
        if isinstance(key, tuple):
            i, j = key
            return self.data[i][j]
        return self.data[key]

    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            i, j = key
            self.data[i][j] = value
        else:
            self.data[key] = value

    def assign(self, other):
        if isinstance(other, matrix):
            if self.n != other.n or self.m != other.m:
                raise ValueError("Matrices must have same dimensions.")
            self.data = [row[:] for row in other.data]

        elif isinstance(other, list):
            if len(other) != self.n or any(len(row) != self.m for row in other):
                raise ValueError("List dimensions must match matrix dimensions.")
            self.data = [row[:] for row in other]

        else:
            raise ValueError("Invalid assignment type.")

    def __repr__(self):
        return "\n".join(str(row) for row in self.data)


In [2]:
M1 = matrix(2, 3)
print("M1 initialized with size:")
print(M1)


M1 initialized with size:
[0, 0, 0]
[0, 0, 0]


In [3]:
M2 = matrix([[1, 2], [3, 4]])
print("\nM2 initialized with list:")
print(M2)



M2 initialized with list:
[1, 2]
[3, 4]


In [4]:
try:
    matrix([[1, 2], [3]])
except ValueError as e:
    print("\nImproper matrix caught:")
    print(e)



Improper matrix caught:
All rows must have the same number of columns.


In [5]:
print("\nAccess using M[i][j]:", M2[0][1])


Access using M[i][j]: 2


In [6]:
print("Access using M[i,j]:", M2[1,0])

Access using M[i,j]: 3


In [7]:
M2[0,0] = 99
print("\nAfter assignment M2[0,0] = 99:")
print(M2)


After assignment M2[0,0] = 99:
[99, 2]
[3, 4]


In [8]:
M3 = matrix(2, 2)
M3.assign(M2)
print("\nM3 after assigning from M2:")
print(M3)


M3 after assigning from M2:
[99, 2]
[3, 4]


In [9]:
M3.assign([[7, 8], [9, 10]])
print("\nM3 after assigning from list:")
print(M3)


M3 after assigning from list:
[7, 8]
[9, 10]


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * (Extra credit) Modify `__getitem__` implemented above to support slicing.


In [10]:
class matrix:
    def __init__(self, *args):
        if len(args) == 2 and all(isinstance(x, int) for x in args):
            n, m = args
            if n <= 0 or m <= 0:
                raise ValueError("Matrix dimensions must be positive.")
            self.n = n
            self.m = m
            self.data = [[0 for _ in range(m)] for _ in range(n)]

        elif len(args) == 1 and isinstance(args[0], list):
            data = args[0]
            if not data or not all(isinstance(row, list) for row in data):
                raise ValueError("Must provide a list of lists.")

            row_len = len(data[0])
            for row in data:
                if len(row) != row_len:
                    raise ValueError("All rows must have same length.")

            self.n = len(data)
            self.m = row_len
            self.data = [row[:] for row in data]

        else:
            raise ValueError("Invalid initialization.")


    def shape(self):
        return (self.n, self.m)

    def transpose(self):
        transposed = [[self.data[i][j] for i in range(self.n)]
                      for j in range(self.m)]
        return matrix(transposed)

    def row(self, n):
        if n < 0 or n >= self.n:
            raise IndexError("Row index out of range.")
        return matrix([self.data[n][:]])   # 1 x m matrix

    def column(self, n):
        if n < 0 or n >= self.m:
            raise IndexError("Column index out of range.")
        col = [[self.data[i][n]] for i in range(self.n)]
        return matrix(col)   # n x 1 matrix

    def to_list(self):
        return [row[:] for row in self.data]

    def block(self, n0, n1, m0, m1):
        if not (0 <= n0 <= n1 <= self.n and 0 <= m0 <= m1 <= self.m):
            raise ValueError("Invalid block indices.")
        sub = [self.data[i][m0:m1] for i in range(n0, n1)]
        return matrix(sub)

##Slicing

    def __getitem__(self, key):
        if isinstance(key, tuple):
            r, c = key

            if isinstance(r, slice) or isinstance(c, slice):
                rows = range(*r.indices(self.n)) if isinstance(r, slice) else [r]
                cols = range(*c.indices(self.m)) if isinstance(c, slice) else [c]
                sub = [[self.data[i][j] for j in cols] for i in rows]
                return matrix(sub)
            else:
                return self.data[r][c]

        return self.data[key]

    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            i, j = key
            self.data[i][j] = value
        else:
            self.data[key] = value

    def __repr__(self):
        return "\n".join(str(row) for row in self.data)


In [15]:
M = matrix([[1,2,3],
            [4,5,6],
            [7,8,9]])

In [14]:
print("Shape:", M.shape())

Shape: (3, 3)


In [16]:
print("\nTranspose:")
print(M.transpose())


Transpose:
[1, 4, 7]
[2, 5, 8]
[3, 6, 9]


In [17]:
print("\nRow 1:")
print(M.row(1))


Row 1:
[4, 5, 6]


In [18]:
print("Row shape:", M.row(1).shape())

Row shape: (1, 3)


In [19]:
print("\nColumn 2:")
print(M.column(2))


Column 2:
[3]
[6]
[9]


In [20]:
print("Column shape:", M.column(2).shape())

Column shape: (3, 1)


In [21]:
print("\nBack to list:")
print(M.to_list())


Back to list:
[[1, 2, 3], [4, 5, 6], [7, 8, 9]]


In [22]:
print("\nBlock (0:2, 1:3):")
print(M.block(0,2,1,3))


Block (0:2, 1:3):
[2, 3]
[5, 6]


In [23]:
print("\nM[1,2]:", M[1,2])


M[1,2]: 6


In [24]:
print("\nM[0:2, 1:3]:")
print(M[0:2,1:3])


M[0:2, 1:3]:
[2, 3]
[5, 6]


In [25]:
print("\nM[:,1]:")
print(M[:,1])


M[:,1]:
[2]
[5]
[8]


3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

In [26]:
def constant(n, m, c):
    if n <= 0 or m <= 0:
        raise ValueError("Dimensions must be positive.")
    return matrix([[float(c) for _ in range(m)] for _ in range(n)])


def zeros(n, m):
    return constant(n, m, 0.0)


def ones(n, m):
    return constant(n, m, 1.0)


def eye(n):
    if n <= 0:
        raise ValueError("Dimension must be positive.")
    data = []
    for i in range(n):
        row = []
        for j in range(n):
            if i == j:
                row.append(1.0)
            else:
                row.append(0.0)
        data.append(row)
    return matrix(data)


In [28]:
A = constant(2,3,5)
print("constant(2,3,5):")
print(A)
print("Shape:", A.shape())

constant(2,3,5):
[5.0, 5.0, 5.0]
[5.0, 5.0, 5.0]
Shape: (2, 3)


In [29]:
Z = zeros(3,2)
print("\nzeros(3,2):")
print(Z)
print("Shape:", Z.shape())


zeros(3,2):
[0.0, 0.0]
[0.0, 0.0]
[0.0, 0.0]
Shape: (3, 2)


In [30]:
O = ones(2,2)
print("\nones(2,2):")
print(O)
print("Shape:", O.shape())


ones(2,2):
[1.0, 1.0]
[1.0, 1.0]
Shape: (2, 2)


In [31]:
I = eye(4)
print("\neye(4):")
print(I)
print("Shape:", I.shape())


eye(4):
[1.0, 0.0, 0.0, 0.0]
[0.0, 1.0, 0.0, 0.0]
[0.0, 0.0, 1.0, 0.0]
[0.0, 0.0, 0.0, 1.0]
Shape: (4, 4)


4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

In [34]:
class matrix:
    def __init__(self, *args):
        if len(args) == 2 and all(isinstance(x, int) for x in args):
            n, m = args
            if n <= 0 or m <= 0:
                raise ValueError("Matrix dimensions must be positive.")
            self.n = n
            self.m = m
            self.data = [[0 for _ in range(m)] for _ in range(n)]

        elif len(args) == 1 and isinstance(args[0], list):
            data = args[0]
            if not data or not all(isinstance(row, list) for row in data):
                raise ValueError("Must provide a list of lists.")

            row_len = len(data[0])
            for row in data:
                if len(row) != row_len:
                    raise ValueError("All rows must have same length.")

            self.n = len(data)
            self.m = row_len
            self.data = [row[:] for row in data]

        else:
            raise ValueError("Invalid initialization.")


    def shape(self):
        return (self.n, self.m)

    def transpose(self):
        transposed = [[self.data[i][j] for i in range(self.n)]
                      for j in range(self.m)]
        return matrix(transposed)

    def row(self, n):
        if n < 0 or n >= self.n:
            raise IndexError("Row index out of range.")
        return matrix([self.data[n][:]])   # 1 x m matrix

    def column(self, n):
        if n < 0 or n >= self.m:
            raise IndexError("Column index out of range.")
        col = [[self.data[i][n]] for i in range(self.n)]
        return matrix(col)   # n x 1 matrix

    def to_list(self):
        return [row[:] for row in self.data]

    def block(self, n0, n1, m0, m1):
        if not (0 <= n0 <= n1 <= self.n and 0 <= m0 <= m1 <= self.m):
            raise ValueError("Invalid block indices.")
        sub = [self.data[i][m0:m1] for i in range(n0, n1)]
        return matrix(sub)

##Slicing

    def __getitem__(self, key):
        if isinstance(key, tuple):
            r, c = key

            if isinstance(r, slice) or isinstance(c, slice):
                rows = range(*r.indices(self.n)) if isinstance(r, slice) else [r]
                cols = range(*c.indices(self.m)) if isinstance(c, slice) else [c]
                sub = [[self.data[i][j] for j in cols] for i in rows]
                return matrix(sub)
            else:
                return self.data[r][c]

        return self.data[key]

    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            i, j = key
            self.data[i][j] = value
        else:
            self.data[key] = value

    def __repr__(self):
        return "\n".join(str(row) for row in self.data)


##

    def scalarmul(self, c):
        return matrix([[self.data[i][j] * c for j in range(self.m)]
                       for i in range(self.n)])

    def add(self, other):
        if self.shape() != other.shape():
            raise ValueError("Matrix dimensions must agree for addition.")
        return matrix([[self.data[i][j] + other.data[i][j]
                        for j in range(self.m)]
                       for i in range(self.n)])

    def sub(self, other):
        if self.shape() != other.shape():
            raise ValueError("Matrix dimensions must agree for subtraction.")
        return matrix([[self.data[i][j] - other.data[i][j]
                        for j in range(self.m)]
                       for i in range(self.n)])

    def mat_mult(self, other):
        if self.m != other.n:
            raise ValueError("Inner matrix dimensions must agree for multiplication.")

        result = []
        for i in range(self.n):
            row = []
            for j in range(other.m):
                total = 0
                for k in range(self.m):
                    total += self.data[i][k] * other.data[k][j]
                row.append(total)
            result.append(row)

        return matrix(result)

    def element_mult(self, other):
        if self.shape() != other.shape():
            raise ValueError("Matrix dimensions must agree for element-wise multiplication.")
        return matrix([[self.data[i][j] * other.data[i][j]
                        for j in range(self.m)]
                       for i in range(self.n)])

    def equals(self, other):
        if self.shape() != other.shape():
            return False
        for i in range(self.n):
            for j in range(self.m):
                if self.data[i][j] != other.data[i][j]:
                    return False
        return True


In [35]:
A = matrix([[1,2],
            [3,4]])

B = matrix([[5,6],
            [7,8]])

In [36]:
print("Scalar multiply 3*A:")
print(A.scalarmul(3))

Scalar multiply 3*A:
[3, 6]
[9, 12]


In [37]:
print("\nA + B:")
print(A.add(B))


A + B:
[6, 8]
[10, 12]


In [38]:
print("\nA - B:")
print(A.sub(B))


A - B:
[-4, -4]
[-4, -4]


In [39]:
print("\nA * B:")
print(A.mat_mult(B))


A * B:
[19, 22]
[43, 50]


In [40]:
C = matrix([[1,2],[3,4]])

print("\nA equals C:", A.equals(C))
print("A equals B:", A.equals(B))


A equals C: True
A equals B: False


In [41]:
print("\nElement-wise multiply A and B:")
print(A.element_mult(B))


Element-wise multiply A and B:
[5, 12]
[21, 32]


In [42]:
print("A equals 3x3 matrix:", A.equals(matrix(3,3)))

A equals 3x3 matrix: False


5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N
    * M=N


In [44]:
class matrix:
    def __init__(self, *args):
        if len(args) == 2 and all(isinstance(x, int) for x in args):
            n, m = args
            if n <= 0 or m <= 0:
                raise ValueError("Matrix dimensions must be positive.")
            self.n = n
            self.m = m
            self.data = [[0 for _ in range(m)] for _ in range(n)]

        elif len(args) == 1 and isinstance(args[0], list):
            data = args[0]
            if not data or not all(isinstance(row, list) for row in data):
                raise ValueError("Must provide a list of lists.")

            row_len = len(data[0])
            for row in data:
                if len(row) != row_len:
                    raise ValueError("All rows must have same length.")

            self.n = len(data)
            self.m = row_len
            self.data = [row[:] for row in data]

        else:
            raise ValueError("Invalid initialization.")


    def shape(self):
        return (self.n, self.m)

    def transpose(self):
        transposed = [[self.data[i][j] for i in range(self.n)]
                      for j in range(self.m)]
        return matrix(transposed)

    def row(self, n):
        if n < 0 or n >= self.n:
            raise IndexError("Row index out of range.")
        return matrix([self.data[n][:]])   # 1 x m matrix

    def column(self, n):
        if n < 0 or n >= self.m:
            raise IndexError("Column index out of range.")
        col = [[self.data[i][n]] for i in range(self.n)]
        return matrix(col)   # n x 1 matrix

    def to_list(self):
        return [row[:] for row in self.data]

    def block(self, n0, n1, m0, m1):
        if not (0 <= n0 <= n1 <= self.n and 0 <= m0 <= m1 <= self.m):
            raise ValueError("Invalid block indices.")
        sub = [self.data[i][m0:m1] for i in range(n0, n1)]
        return matrix(sub)

##Slicing

    def __getitem__(self, key):
        if isinstance(key, tuple):
            r, c = key

            if isinstance(r, slice) or isinstance(c, slice):
                rows = range(*r.indices(self.n)) if isinstance(r, slice) else [r]
                cols = range(*c.indices(self.m)) if isinstance(c, slice) else [c]
                sub = [[self.data[i][j] for j in cols] for i in rows]
                return matrix(sub)
            else:
                return self.data[r][c]

        return self.data[key]

    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            i, j = key
            self.data[i][j] = value
        else:
            self.data[key] = value

    def __repr__(self):
        return "\n".join(str(row) for row in self.data)


##

    def scalarmul(self, c):
        return matrix([[self.data[i][j] * c for j in range(self.m)]
                       for i in range(self.n)])

    def add(self, other):
        if self.shape() != other.shape():
            raise ValueError("Matrix dimensions must agree for addition.")
        return matrix([[self.data[i][j] + other.data[i][j]
                        for j in range(self.m)]
                       for i in range(self.n)])

    def sub(self, other):
        if self.shape() != other.shape():
            raise ValueError("Matrix dimensions must agree for subtraction.")
        return matrix([[self.data[i][j] - other.data[i][j]
                        for j in range(self.m)]
                       for i in range(self.n)])

    def mat_mult(self, other):
        if self.m != other.n:
            raise ValueError("Inner matrix dimensions must agree for multiplication.")

        result = []
        for i in range(self.n):
            row = []
            for j in range(other.m):
                total = 0
                for k in range(self.m):
                    total += self.data[i][k] * other.data[k][j]
                row.append(total)
            result.append(row)

        return matrix(result)

    def element_mult(self, other):
        if self.shape() != other.shape():
            raise ValueError("Matrix dimensions must agree for element-wise multiplication.")
        return matrix([[self.data[i][j] * other.data[i][j]
                        for j in range(self.m)]
                       for i in range(self.n)])

    def equals(self, other):
        if self.shape() != other.shape():
            return False
        for i in range(self.n):
            for j in range(self.m):
                if self.data[i][j] != other.data[i][j]:
                    return False
        return True

##overload

    def __add__(self, other):
        return self.add(other)

    def __sub__(self, other):
        return self.sub(other)

    def __mul__(self, other):
        # Scalar multiplication
        if isinstance(other, (int, float)):
            return self.scalarmul(other)

        # Matrix multiplication
        if isinstance(other, matrix):
            return self.mat_mult(other)

        raise TypeError("Unsupported operand type for *")

    def __rmul__(self, other):
        # Allows 2*M
        if isinstance(other, (int, float)):
            return self.scalarmul(other)
        raise TypeError("Unsupported operand type for *")

    def __eq__(self, other):
        return self.equals(other)


In [45]:
A = matrix([[1,2],
            [3,4]])

B = matrix([[5,6],
            [7,8]])

C = matrix([[1,2],
            [3,4]])

In [46]:
print("2*A:")
print(2*A)

2*A:
[2, 4]
[6, 8]


In [47]:
print("\nA*2:")
print(A*2)


A*2:
[2, 4]
[6, 8]


In [48]:
print("\nA+B:")
print(A+B)


A+B:
[6, 8]
[10, 12]


In [49]:
print("\nA-B:")
print(A-B)


A-B:
[-4, -4]
[-4, -4]


In [50]:
print("\nA*B:")
print(A*B)


A*B:
[19, 22]
[43, 50]


In [51]:
print("\nA == C:", A == C)
print("A == B:", A == B)


A == C: True
A == B: False


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [52]:
A = matrix([[1, 2],
            [3, 4]])

B = matrix([[2, 0],
            [1, 2]])

C = matrix([[0, 1],
            [1, 0]])

I = eye(2)

In [53]:
left  = (A * B) * C
right = A * (B * C)

print("(AB)C =")
print(left)

print("\nA(BC) =")
print(right)

print("\nAssociativity holds:", left == right)

(AB)C =
[4, 4]
[8, 10]

A(BC) =
[4, 4]
[8, 10]

Associativity holds: True


In [54]:
left  = A * (B + C)
right = (A * B) + (A * C)

print("A(B+C) =")
print(left)

print("\nAB + AC =")
print(right)

print("\nDistributive property holds:", left == right)

A(B+C) =
[6, 5]
[14, 11]

AB + AC =
[6, 5]
[14, 11]

Distributive property holds: True


In [55]:
AB = A * B
BA = B * A

print("AB =")
print(AB)

print("\nBA =")
print(BA)

print("\nCommutative?", AB == BA)

AB =
[4, 4]
[10, 8]

BA =
[2, 4]
[7, 10]

Commutative? False


In [56]:
AI = A * I

print("A =")
print(A)

print("\nAI =")
print(AI)

print("\nIdentity property holds:", AI == A)

A =
[1, 2]
[3, 4]

AI =
[1.0, 2.0]
[3.0, 4.0]

Identity property holds: True
